In [1]:
import pandas as pd
import json
import pyarrow.parquet as pq
from pathlib import Path

In [2]:
systems_cleaned = pd.read_csv('../../data/core/systems_cleaned.csv')

## Prize data exploration

Find all module types *and* see if any system has multiple module types.

In [30]:
prize_module_types = set()

In [31]:
prize_system_ids = [2105, 2107, 7333, 9068, 9069]
for system_id in prize_system_ids:
    # load the metadata
    metadata_filepath = Path(
            '../../data/raw/prize-metadata/'
            + f'{system_id}_system_metadata.json'
        )
    with open(metadata_filepath) as json_reader:
        local_metadata = json.load(json_reader)
        system_modules = local_metadata['Modules']
        local_module_types = []
        for key in system_modules.keys():
            local_module_types.append(system_modules[key]['type'])
        local_module_types = set(local_module_types)
        if len(local_module_types) > 1:
            print(f'Multiple module types for system {system_id}')
        prize_module_types = prize_module_types.union(local_module_types)
prize_module_types

{'CdTe', 'mono-Si', 'multi-Si'}

No multiple types, yay!

## Parquet data

In [5]:
modules_dir = Path('../../data/raw/parquet-modules/')
modules_pq = pq.ParquetDataset(modules_dir)
modules_df = modules_pq.read().to_pandas()
modules_df.head()


,comments,end_on,inverter_id,manufacturer,model,module_id,name,quantity,reference_module,serial_num,start_on,type,site_id,system_id
0,,,14539,Sharp,NU-U235F2,12498,mod_nist_r1,312,False,,2014-07-29 07:59:00,mono-Si,3198,4903
1,,,14537,Sharp,NU-U235F2,12496,mod_nist_c1,1032,False,,2014-07-29 07:59:00,mono-Si,3196,4901
2,,,14538,Sharp,NU-U235F2,12497,mod_nist_g1,1152,False,,2014-07-29 07:59:00,mono-Si,3197,4902
3,,,1317,SunPower,e18/230,1337,re-no-1-neme-diano_1344_m0,20,False,,2012-02-28 18:52:00,mono-Si,1325,1344
4,,,1229,SunPower,e18/230,1246,re-no-1-diano_1254_m0,20,False,,2018-03-31 11:23:10,mono-Si,1242,1254


Again, look at types

In [6]:
parquet_module_types = set(modules_df['type'])
parquet_module_types

{'SJT',
 'Unknown',
 'amorphous si',
 'cis family thin-film',
 'cylindrical cigs',
 'mono-Si',
 'multi-Si',
 'ribbon polycrystalline si'}

Check for multiples.

In [37]:
for system_id in modules_df['system_id'].unique():
    local_view = modules_df[modules_df['system_id'] == system_id]
    local_types = []
    for ind in local_view.index:
        local_types.append(local_view.loc[ind, 'type'])
    local_types = set(local_types)
    if len(local_types) > 1:
        print(f'System {system_id} has multiple types of solar cell.')

No multiples, yay!

## CSV Group

In [38]:
csv_module_types = set()

In [41]:
csv_metadata_dir = Path('../../data/raw/csv-metadata/')
jsons = csv_metadata_dir.glob("*_system_metadata.json")
for file_path in jsons:
    system_id = int(
        file_path.parts[-1].replace('_system_metadata.json', '')
    )
    with open(file_path) as reader:
        local_metadata = json.load(reader)
        has_modules = True
        try:
            system_modules = local_metadata['Modules']
        except KeyError:
            has_modules = False
        except BaseException as e:
            raise e
    if has_modules:
        local_module_types = []
        for key in system_modules.keys():
            local_module_types.append(system_modules[key]['type'])
        local_module_types = set(local_module_types)
        if len(local_module_types) > 1:
            print(f'System {system_id} has multiple module types:')
            for item in local_module_types:
                print(item)
        csv_module_types = csv_module_types.union(local_module_types)
    else:
        print(f"System {system_id} has no module types!")
csv_module_types

{'HIT',
 'PERC',
 'PERC Bifacial',
 'PERC IBC',
 'SJT',
 'Unknown',
 'amorphous si',
 'cis family thin-film',
 'cylindrical cigs',
 'mono-Si',
 'mono-Si IBC',
 'multi-Si',
 'n-PERT',
 'n-PERT IBC',
 'n-SHJ',
 'n-TOPCon',
 'ribbon polycrystalline si'}

No multiple types, yay!

## Put it all together

In [11]:
all_module_types = csv_module_types.union(prize_module_types, parquet_module_types)
all_module_types

{'CdTe',
 'HIT',
 'PERC',
 'PERC Bifacial',
 'PERC IBC',
 'SJT',
 'Unknown',
 'amorphous si',
 'cis family thin-film',
 'cylindrical cigs',
 'mono-Si',
 'mono-Si IBC',
 'multi-Si',
 'n-PERT',
 'n-PERT IBC',
 'n-SHJ',
 'n-TOPCon',
 'ribbon polycrystalline si'}

As per https://www.energy.gov/eere/solar/solar-photovoltaic-cell-basics,
'cigs' and 'CdTe' are both variants of thin-film.
As per https://en.wikipedia.org/wiki/Thin-film_solar_cell, amorphous silicon is also thin-film.

As per https://en.wikipedia.org/wiki/Polycrystalline_silicon,
polycrystalline and multi-crystalline structures are two names for the same thing, so multi-SI and poly-Si 
are mostly the same thing.

As per https://en.wikipedia.org/wiki/Heterojunction_solar_cell, HIT and SHJ are a hybrid between thin-film and regular Si cells.

As per https://www.ines-solaire.org/en/research-innovation/photovoltaique-haut-rendement/, SJT is another variant for SJT/HIT.

As per (attach article here), PERC, PERT and IBC methods are silicon solar cells with extra tricks on the back layer to improve efficiency.
As per https://b2b.technosun.com/en/blog/professional-photovoltaic-area-1/perc-perl-and-pert-cells-135, PERC is a mono-SI variant and PERT is a multi-SI variant.
BUT see https://solarmagazine.com/solar-panels/perc-solar-panels/ and https://www.ecoflow.com/us/blog/types-of-solar-panels, which claims that PERC can be used with both mone and multi!

As per https://www.solarnplus.com/topcon-cell-technology-what-is-it-and-how-it-works/,  and https://pv-manufacturing.org/tunnel-oxide-passivated-contact-topcon-solar-cells/, TOPCon is alos a poly-SI variant.


In [26]:
def simplified_classifier(cell_type: str, raise_errors=True):
    cell_type = cell_type.lower()
    if ('mono' in cell_type):
        return 'monocrystalline_Si'
    elif ('poly' in cell_type) or ('multi' in cell_type):
        return 'multicrystalline_Si'
    elif ('perc' in cell_type) or ('pert' in cell_type):
        return 'some_crystalline_Si_with_passivation'
    elif ('cigs' in cell_type) or ('cdte' in cell_type)\
     or ('thin' in cell_type) or ('amorphous' in cell_type):
        return 'thin_film'
    elif ('hit' in cell_type) or ('sjt' in cell_type)\
     or ('shj' in cell_type) or ('topcon' in cell_type):
        return 'heterojunction'
    elif ('unknown' in cell_type):
        return 'unknown'
    elif raise_errors:
        raise ValueError(f'Missing classification! for {cell_type}')
    else:
        return 'unclassified'

In [40]:
for visited_type in all_module_types:
    print(f'{visited_type} is classified as '\
          + simplified_classifier(visited_type, True) + '.')

n-PERT IBC is classified as some_crystalline_Si_with_passivation.
HIT is classified as heterojunction.
ribbon polycrystalline si is classified as multicrystalline_Si.
SJT is classified as heterojunction.
mono-Si is classified as monocrystalline_Si.
cis family thin-film is classified as thin_film.
CdTe is classified as thin_film.
n-SHJ is classified as heterojunction.
n-TOPCon is classified as heterojunction.
PERC Bifacial is classified as some_crystalline_Si_with_passivation.
mono-Si IBC is classified as monocrystalline_Si.
Unknown is classified as unknown.
PERC is classified as some_crystalline_Si_with_passivation.
PERC IBC is classified as some_crystalline_Si_with_passivation.
amorphous si is classified as thin_film.
multi-Si is classified as multicrystalline_Si.
cylindrical cigs is classified as thin_film.
n-PERT is classified as some_crystalline_Si_with_passivation.
